In [3]:
import torch
import torch.nn as nn

# ---------------------------
# 1. Toy Dataset
# ---------------------------

In [4]:
sentences = [
    "i love pizza",
    "this is awful",
    "i hate everything",
    "such a great movie"
]

labels = torch.tensor([1, 0, 0, 1])  # sentiment labels

# ---------------------------
# 2. Vocabulary
# ---------------------------

In [14]:
word2idx = {"<PAD>": 0}
idx = 1

for sent in sentences:
    for w in sent.split():
        if w not in word2idx:
            word2idx[w] = idx
            idx += 1

vocab_size = len(word2idx)
vocab_size

13

In [16]:
def encode(sentence):
    return torch.tensor([word2idx[w] for w in sentence.split()])

encoded = [encode(s) for s in sentences]
encoded

[tensor([1, 2, 3]),
 tensor([4, 5, 6]),
 tensor([1, 7, 8]),
 tensor([ 9, 10, 11, 12])]

In [17]:
# Pad sentences to equal length
from torch.nn.utils.rnn import pad_sequence

padded = pad_sequence(encoded, batch_first=True)
print("Padded shape:", padded.shape)  # (batch, seq)

Padded shape: torch.Size([4, 4])


In [18]:
padded

tensor([[ 1,  2,  3,  0],
        [ 4,  5,  6,  0],
        [ 1,  7,  8,  0],
        [ 9, 10, 11, 12]])

# ---------------------------
# 3. LSTM Model
# ---------------------------

In [10]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        """
        x shape: (batch, seq_len)
        """
        emb = self.embedding(x)
        """
        emb shape: (batch, seq_len, embed_dim)
        """

        output, (h_n, c_n) = self.lstm(emb)
        """
        output shape: (batch, seq_len, hidden_dim)
        h_n shape: (1, batch, hidden_dim)
        c_n shape: (1, batch, hidden_dim)
        """

        final_hidden = h_n[-1]  # take last-layer hidden state
        logits = self.fc(final_hidden)
        return logits


# ---------------------------
# 4. Training Step
# ---------------------------

In [11]:
model = SentimentLSTM(vocab_size, embed_dim=16, hidden_dim=32, num_classes=2)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [12]:
for epoch in range(50):
    optimizer.zero_grad()

    logits = model(padded)
    loss = criterion(logits, labels)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss = {loss.item():.4f}")

Epoch 0, Loss = 0.6965
Epoch 10, Loss = 0.0157
Epoch 20, Loss = 0.0003
Epoch 30, Loss = 0.0001
Epoch 40, Loss = 0.0000


# ---------------------------
# 5. Prediction Example
# ---------------------------

In [13]:
test = encode("i love movie")
test = test.unsqueeze(0)  # batch dimension

with torch.no_grad():
    pred = torch.argmax(model(test), dim=1).item()

print("Prediction:", pred)

Prediction: 1
